# DCGAN (Deep Convolutional GAN)

This notebook demonstrates a **minimal DCGAN implementation in PyTorch** using the MNIST dataset.

We focus on:
- Clear architecture
- Step-by-step training logic
- Educational comments

This is intended for learning, not for production optimization.

## 1. Imports and Setup

In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


## 2. Hyperparameters

In [2]:

batch_size = 128
epochs = 20
lr = 0.0002

latent_dim = 100
image_channels = 1
image_size = 28


## 3. Dataset and DataLoader
We normalize images to [-1, 1] because the generator uses Tanh activation.

In [3]:

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


## 4. Generator Network (DCGAN Style)
Uses ConvTranspose2d to upsample noise into images.

In [4]:

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 7, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, image_channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


## 5. Discriminator Network (DCGAN Style)
Uses convolution layers to classify real vs fake images.

In [5]:

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(image_channels, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 1, 7, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)


## 6. Model Initialization

In [6]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

G = Generator().to(device)
D = Discriminator().to(device)


## 7. Loss and Optimizers

In [7]:

criterion = nn.BCELoss()

optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))


## 8. Training Loop

In [8]:

for epoch in range(epochs):
    for real_images, _ in loader:
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # Train Discriminator
        noise = torch.randn(batch_size, latent_dim, 1, 1).to(device)
        fake_images = G(noise)

        loss_real = criterion(D(real_images), real_labels)
        loss_fake = criterion(D(fake_images.detach()), fake_labels)
        loss_D = loss_real + loss_fake

        optimizer_D.zero_grad()
        loss_D.backward()
        optimizer_D.step()

        # Train Generator
        loss_G = criterion(D(fake_images), real_labels)

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{epochs}] Loss D: {loss_D.item():.4f}, Loss G: {loss_G.item():.4f}")


Epoch [1/20] Loss D: 0.0002, Loss G: 9.4249


KeyboardInterrupt: 

## 9. Generate Sample Images

In [ ]:

G.eval()
with torch.no_grad():
    z = torch.randn(16, latent_dim, 1, 1).to(device)
    samples = G(z).cpu()

samples = (samples + 1) / 2  # Denormalize
grid = torchvision.utils.make_grid(samples, nrow=4)
plt.imshow(grid.permute(1, 2, 0))
plt.axis('off')


## 10. Key Learning Points
- DCGAN uses convolution instead of fully connected layers
- BatchNorm stabilizes training
- Tanh + normalized inputs are critical
- Training remains adversarial and unstable by nature

Next steps: WGAN, Conditional GAN, StyleGAN